In [4]:
from huggingface_hub import DatasetCard


card = DatasetCard.load(root)

card.data['configs']

README.md: 0.00B [00:00, ?B/s]

[{'config_name': 'full_patient_state',
  'data_files': [{'split': 'full_patient_state',
    'path': 'full_patient_state/*.parquet'}]}]

- need to add some sort of death indicator for reward/penalty

In [87]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.model_selection import StratifiedShuffleSplit

from datasets import load_dataset

pd.set_option("display.max_columns", 136)

root = "ADS599-Capstone/modeling_data"

df_patient = load_dataset(path=root, name='full_patient_state', split='full_patient_state').to_pandas()

In [26]:
df_patient.head()

,ed_stay_id,subject_id,hadm_id,time,Chemistry-Blood,Hematology-Blood,Hematology-Urine,Blood Gas-Blood,Chemistry-Urine,Chemistry-Other Body Fluid,Hematology-Cerebrospinal Fluid,Chemistry-Cerebrospinal Fluid,Hematology-Ascites,Chemistry-Ascites,Chemistry-Pleural,Hematology-Pleural,Hematology-Joint Fluid,Chemistry-Stool,Hematology-Other Body Fluid,Blood Gas-Other Body Fluid,Hematology-Bone Marrow,Hematology-Stool,Chemistry-Joint Fluid,URINE,Rapid Respiratory Viral Screen & Culture,PERITONEAL FLUID,BLOOD CULTURE,STOOL,OTHER,PLEURAL FLUID,TISSUE,SEROLOGY/BLOOD,IMMUNOLOGY,MRSA SCREEN,SWAB,JOINT FLUID,SPUTUM,"FLUID,OTHER",ABSCESS,BRONCHOALVEOLAR LAVAGE,CSF;SPINAL FLUID,Blood (CMV AB),Blood (EBV),BILE,Other,Analgesic - Opioid/NSAID,Antiemetic,Analgesic - Acetaminophen,Antibiotic,Benzodiazepine - Sedative/Anxiolytic,Analgesic - NSAID,Bronchodilator,Antiplatelet,GI - Acid Suppression,Corticosteroid,Beta Blocker,Diuretic,Antipsychotic,Anticoagulant,Insulin/Glucose,Calcium Channel Blocker,Nitrate,ACE Inhibitor,IV Fluid,Anticonvulsant,Antiarrhythmic,weight,ecg_status,rad_status,in_ed,in_ward,ed_boarding,terminal_event,gender,anchor_age,arrival_transport,acuity,height,recon_ace_inhibitor,recon_analgesic___nsaid,recon_analgesic___opioid_nsaid,recon_antiarrhythmic,recon_antibiotic,recon_anticoagulant,recon_anticonvulsant,recon_antiemetic,recon_antiplatelet,recon_antipsychotic,recon_benzodiazepine___sedative_anxiolytic,recon_beta_blocker,recon_bronchodilator,recon_calcium_channel_blocker,recon_corticosteroid,recon_diuretic,recon_gi___acid_suppression,recon_insulin_glucose,recon_nitrate,admission_type,current_temperature,current_heartrate,current_resprate,current_o2sat,current_sbp,current_dbp,current_pain,current_pulse_pressure,current_map,time_since_last_hrs,temperature_rolling2h,temperature_delta,temperature_rate,heartrate_rolling2h,heartrate_delta,heartrate_rate,resprate_rolling2h,resprate_delta,resprate_rate,o2sat_rolling2h,o2sat_delta,o2sat_rate,sbp_rolling2h,sbp_delta,sbp_rate,dbp_rolling2h,dbp_delta,dbp_rate,observe,dispense_meds,ward_transfer,discharge,transfer_icu,ecg_ordered,rad_ordered,vitals_checked,labs_ordered,micro_ordered
0,30000012,11714491,21562392.0,2126-02-14 20:22:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,1,0,0,DISCHARGE_WARD,F,65,AMBULANCE,2.000000000,65.0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,1,0,0,NaN,98.8,96.0,18.0,93.0,160.0,54.0,0.0,106.0,89.33,0.0,98.8,0.0,NaN,96.0,0.0,NaN,18.0,0.0,NaN,93.0,0.0,NaN,160.0,0.0,NaN,54.0,0.0,NaN,0,0,0,0,0,0,0,1.0,0,0
1,30000012,11714491,21562392.0,2126-02-14 20:52:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,3.0,0.0,1,0,0,DISCHARGE_WARD,F,65,AMBULANCE,2.000000000,65.0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,1,0,0,NaN,98.8,96.0,18.0,93.0,160.0,54.0,0.0,106.0,89.33,0.5,98.8,0.0,NaN,96.0,0.0,NaN,18.0,0.0,NaN,93.0,0.0,NaN,160.0,0.0,NaN,54.0,0.0,NaN,0,0,0,0,0,1,0,0.0,0,0
2,30000012,11714491,21562392.0,2126-02-14 21:22:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,3.0,0.0,1,0,0,DISCHARGE_WARD,F,65,AMBULANCE,2.000000000,65.0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,1,0,0,NaN,98.8,96.0,18.0,93.0,160.0,54.0,0.0,106.0,89.33,1.0,98.8,0.0,NaN,96.0,0.0,NaN,18.0,0.0,NaN,93.0,0.0,NaN,160.0,0.0,NaN,54.0,0.0,NaN,1,0,0,0,0,0,0,0.0,0,0
3,30000012,11714491,21562392.0,2126-02-14 21:52:00,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0

# Preprocess and One-Hot-Encode Categorical Columns

In [88]:
# Map Gender
gender_map = {"F": 1, "M": 0}
df_patient['gender'] = df_patient['gender'].map(gender_map)

# Change acuity to integer
df_patient['acuity'] = df_patient['acuity'].astype(int)

# Mask height and weight
df_patient['height_missing'] = df_patient['height'].isna().astype(int)
df_patient['weight_missing'] = df_patient['weight'].isna().astype(int)

df_patient[['height', 'weight']] = df_patient[['height', 'weight']].fillna(0)

# Create pain_missing column and convert the Other category to 0
df_patient['pain_missing'] = (df_patient['current_pain'] == 'Other').astype(int)
df_patient['current_pain'] = pd.to_numeric(df_patient['current_pain'], errors='coerce').fillna(0)

# Mask admission type then one hot encode admission and arrival
df_patient['admission_missing'] = df_patient['admission_type'].isna().astype(int)
at_dummies = pd.get_dummies(df_patient['admission_type'], prefix='admission_type', dummy_na=False, dtype=int)
arrival_dummies = pd.get_dummies(df_patient['arrival_transport'], prefix='arrival_transport', dtype=int)
df_patient_updated = pd.concat([df_patient, at_dummies, arrival_dummies], axis=1).drop(columns=['admission_type', 'arrival_transport'])

# Isolate Columns and Fill in Config

In [89]:
# Columns out of order so this pieces all the state cols together
vitals = [c for c in df_patient_updated.loc[:, "current_temperature":"dbp_rate"].columns if not c.endswith("_rate") and not c.endswith('_delta')]
med_cols = [c for c in df_patient_updated.columns if c.startswith('recon')]
admission_cols = at_dummies.columns.to_list() + ['admission_missing']
arrival_cols = arrival_dummies.columns.to_list()

# Definitive list of patient state cols
state_cols = df_patient_updated.columns[4:71].to_list() + ['gender', 'anchor_age', 'acuity', 'height', 'height_missing', 'weight_missing', 'pain_missing'] + med_cols + vitals + admission_cols + arrival_cols

# Action cols to be used as labels for supervisor network
action_cols = ['observe', 'vitals_checked', 'labs_ordered', 'micro_ordered', 'ecg_ordered', 'rad_ordered', 'dispense_meds', 'ward_transfer', 'discharge', 'transfer_icu']

Not including the rate and delta columns calculated from vitals until we can ensure they were calculated properly.  Too many NaN values in those columns at the moment

In [90]:
CONFIG = {

    # ── Data ──────────────────────────────────────────────────────────────────
    "state_cols": state_cols, # list of preprocessed feature column names

    "action_cols": action_cols,  # column name for the action taken

    "reward_col": "terminal_event",  # column name for the observed reward
    
    "next_state_cols":  [],         # column names for the next state
      
    "done_col": ['discharge', 'transfer_icu'], # columns indicating episode termination

    # ── Action space ──────────────────────────────────────────────────────────
    "n_actions": 10,       # number of discrete action e.g. 2 for binary (transfer / don't transfer)

    # ── Network architecture ───────────────────────────────────────────────────
    "hidden_layers": [256, 256], # hidden layer sizes for all networks
                                    # BCQ and distributional nets share this
    "activation": "relu", 
    
    # ── BCQ-specific ──────────────────────────────────────────────────────────
    "latent_dim":       32,         # VAE latent space dimensionality
                                    # typically state_dim // 2 is a good start
    "n_candidates":     10,         # number of actions sampled from VAE
                                    # per forward pass at inference
                                    # higher = better coverage, slower inference
    "vae_beta":         0.5,        # KL divergence weight in VAE loss
                                    # higher = more regularized latent space
    "vae_kl_clip":      0.5,        # clamp on VAE latent std to prevent
                                    # posterior collapse during early training
    "perturbation_phi": 0.05,       # max perturbation magnitude applied to
                                    # VAE-sampled actions by the perturbation net
                                    # smaller = stays closer to behavior policy
        # ── Distributional Q-specific (QR-DQN) ────────────────────────────────────
    "n_quantiles":      51,         # number of quantile atoms
                                    # more = finer return distribution
                                    # 51 (C51) and 200 (QR-DQN) are common choices
    "huber_kappa":      1.0,        # Huber loss threshold in quantile regression
                                    # controls transition between L1 and L2 loss

    # ── Training ──────────────────────────────────────────────────────────────
    "learning_rate":    1e-4,       # shared learning rate for all optimizers
    "vae_learning_rate":1e-3,       # VAE can tolerate a higher learning rate
                                    # than the Q-network
    "batch_size":       64,
    "n_epochs":         100,
    "gamma":            0.99,       # discount factor
    "tau":              0.005,      # soft target network update rate
                                    # BCQ typically uses smaller tau than CQL
    "grad_clip":        1.0,        # gradient clipping max norm (None to disable)

    # ── Regularization ────────────────────────────────────────────────────────
    "dropout":          0.0,        # dropout between hidden layers (0 = off)
    "weight_decay":     1e-4,       # L2 regularization on Q-network optimizer

    # ── Risk preference (distributional) ──────────────────────────────────────
    "risk_measure":     "mean",     # how to select actions from the distribution
                                    # "mean"  — maximize expected return (neutral)
                                    # "cvar"  — minimize conditional value at risk
                                    #           (conservative, good for clinical)
                                    # "worst" — maximize worst-case quantile
    "cvar_alpha":       0.25,       # CVaR tail fraction (only used if
                                    # risk_measure = "cvar")
                                    # 0.25 = focus on bottom 25% of outcomes

    # ── Reproducibility ───────────────────────────────────────────────────────
    "seed":             10,

    # ── Output ────────────────────────────────────────────────────────────────
    "model_save_path":  "bcq_distributional.pt",
    "log_every_n":      10,
}

# Split Data and Scale Numerical Columns

In [91]:
stay_actions = (
    df_patient_updated.groupby("ed_stay_id")[action_cols]
    .max()
    .reset_index()
)

# Train
train_split = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=CONFIG['seed'])
resample_idx, sample_idx = next(train_split.split(stay_actions, stay_actions[action_cols]))
sample_stays = set(stay_actions.iloc[sample_idx]['ed_stay_id'])
df_train = df_patient_updated[df_patient_updated['ed_stay_id'].isin(sample_stays)].copy()

# Test
stay_test = stay_actions.loc[resample_idx]
test_split = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.08, random_state=CONFIG['seed'])
_, sample_idx = next(test_split.split(stay_test, stay_test[action_cols]))
sample_stays = set(stay_test.iloc[sample_idx]['ed_stay_id'])
df_test = df_patient_updated[df_patient_updated['ed_stay_id'].isin(sample_stays)].copy()

print(f"Sample stays: {df_train['ed_stay_id'].nunique():,}")
print(df_train.drop_duplicates('ed_stay_id')['terminal_event'].value_counts())
print(f"\nSample stays: {df_test['ed_stay_id'].nunique():,}")
print(df_test.drop_duplicates('ed_stay_id')['terminal_event'].value_counts())

train_idx = set(df_train.index)
test_idx = set(df_test.index)

print(f"\nOverlap: {train_idx.intersection(test_idx)}")

Sample stays: 8,422
terminal_event
DISCHARGE_ED      3814
DISCHARGE_WARD    2783
ICU               1825
Name: count, dtype: int64

Sample stays: 6,064
terminal_event
DISCHARGE_ED      2746
DISCHARGE_WARD    2004
ICU               1314
Name: count, dtype: int64

Overlap: set()


## Scaling

In [92]:
scale = StandardScaler()
scaling_cols = vitals + ['anchor_age', 'weight', 'height', 'time_since_last_hrs']

In [93]:
df_train[scaling_cols] = scale.fit_transform(df_train[scaling_cols])
df_test[scaling_cols] = scale.transform(df_test[scaling_cols])

In [95]:
missing = [c for c in state_cols if c not in df_train.columns]
extra = [c for c in df_train.columns if c not in state_cols + action_cols + ['ed_stay_id', 'subject_id', 'hadm_id', 'time', 'terminal_event', 'ed_boarding']]
print(f"In state_cols but not in df_train: {missing}")
print(f"In df_train but not accounted for: {extra}")

In state_cols but not in df_train: []
In df_train but not accounted for: ['temperature_delta', 'temperature_rate', 'heartrate_delta', 'heartrate_rate', 'resprate_delta', 'resprate_rate', 'o2sat_delta', 'o2sat_rate', 'sbp_delta', 'sbp_rate', 'dbp_delta', 'dbp_rate']


In [96]:
df_train.drop(columns=extra, inplace=True)
df_test.drop(columns=extra, inplace=True)

# Convert To Tensors

In [ ]:
torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])

def df_to_tensors(df):
    s  = torch.tensor(df[CONFIG["state_cols"]].values.astype(np.float32))
    a  = torch.tensor(df[CONFIG["action_cols"]].values.astype(np.int64))
    r  = torch.tensor(df[CONFIG["reward_col"]].values.astype(np.float32))
    ns = torch.tensor(df[CONFIG["next_state_cols"]].values.astype(np.float32))
    d  = torch.tensor(df[CONFIG["done_col"]].values.astype(np.float32))
    return s, a, r, ns, d

s_train, a_train, r_train, ns_train, d_train = df_to_tensors(df_train)
# s_val,   a_val,   r_val,   ns_val,   d_val   = df_to_tensors(df_val)
s_test,  a_test,  r_test,  ns_test,  d_test  = df_to_tensors(df_test)

train_loader = DataLoader(
    TensorDataset(s_train, a_train, r_train, ns_train, d_train),
    batch_size=CONFIG["batch_size"],
    shuffle=True,
)